# ==============================================================================
# 💳 FINANCIAL UNDERWRITING: CREDIT RISK LOAN DEFAULT VIA RANDOM FOREST
# ==============================================================================
### Core Principles:
Credit risk modeling involves applicant demographics, loan attributes, and default history.
With 32,000 borrower records, Random Forest builds an ensemble of robust trees that evaluate loan default probabilities while mitigating individual tree variance.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Credit Underwriting Analytics Environment ready.")


✅ STEP 1: Credit Underwriting Analytics Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle Credit Risk Dataset (32,581 loan applications).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/credit_risk/credit_risk_dataset.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} applications, {df.shape[1]} attributes")
print(f"Target loan_status: {df['loan_status'].value_counts().to_dict()}")
df.head(3)


📊 Dataset Shape: 32,581 applications, 12 attributes
Target loan_status: {0: 25473, 1: 7108}


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `loan_status` (1 = Default, 0 = Non-Default).


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['loan_status'])
y = df['loan_status']

print(f"Missing attributes:\n{X.isnull().sum()[X.isnull().sum() > 0]}")
print(f"Default Prevalence: {y.mean()*100:.2f}%")


Missing attributes:
person_emp_length     895
loan_int_rate        3116
dtype: int64
Default Prevalence: 21.82%


## 🔒 STEP 6: STRATIFIED SPLIT
80/20 train/test split.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Borrowers: {X_train.shape[0]:,}")
print(f"Test Borrowers : {X_test.shape[0]:,}")


Train Borrowers: 26,064
Test Borrowers : 6,517


## ⚙️ STEP 7: SCALE-INVARIANT PREPROCESSOR
Imputing missing income and employment length, and one-hot encoding loan intent and grade.


In [5]:
# ==============================================================================
# ⚙️ STEP 7: TREE-OPTIMIZED UNIVERSAL PREPROCESSOR (SCALE-INVARIANT)
# ==============================================================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def build_tree_preprocessor(X):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    transformers = []
    if len(num_cols) > 0:
        transformers.append(('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols))
    if len(cat_cols) > 0:
        transformers.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
        ]), cat_cols))
    return ColumnTransformer(transformers=transformers, remainder='drop') if len(transformers) > 0 else 'passthrough'

preprocessor = build_tree_preprocessor(X_train)
print(f"✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.")


✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.


## ⚠️ STEP 8: UNCONSTRAINED BASELINE FIT (OVERFITTING PROOF) & OOB SCORE
Evaluating baseline Random Forest on loan defaults.


In [6]:
# STEP 8: UNCONSTRAINED BASELINE FIT (OVERFITTING PROOF)
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=12, oob_score=True, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)

roc_base = roc_auc_score(y_test, pipe_rf.predict_proba(X_test)[:, 1])
print(f"Baseline Random Forest ROC-AUC : {roc_base*100:.2f}%")
print(f"Out-of-Bag (OOB) Accuracy Score: {pipe_rf.named_steps['rf'].oob_score_*100:.2f}%")


Baseline Random Forest ROC-AUC : 92.44%
Out-of-Bag (OOB) Accuracy Score: 92.82%


## 🎯 STEP 9: HYPERPARAMETER TUNING VIA 5-FOLD CV
Optimizing `max_depth` and `min_samples_leaf` for financial ROC-AUC.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [10, 14],
    'rf__min_samples_leaf': [2, 5]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
])
grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'rf__max_depth': 14, 'rf__min_samples_leaf': 2, 'rf__n_estimators': 150}
🎯 Best 3-Fold CV ROC-AUC: 92.63%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Tree)
Evaluating ensemble ROC-AUC against a single decision tree.


In [8]:
# STEP 10: BENCHMARK
from sklearn.tree import DecisionTreeClassifier

pipe_single = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=42))
])
pipe_single.fit(X_train, y_train)

roc_single = roc_auc_score(y_test, pipe_single.predict_proba(X_test)[:, 1])
roc_rf = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])

print("="*65)
print(f"Single Decision Tree ROC-AUC: {roc_single*100:.2f}%")
print(f"Random Forest (Tuned) ROC-AUC: {roc_rf*100:.2f}% (+{(roc_rf - roc_single)*100:.2f}% ROC gain!)")
print("="*65)


Single Decision Tree ROC-AUC: 84.58%
Random Forest (Tuned) ROC-AUC: 92.62% (+8.04% ROC gain!)


## 📊 STEP 11: FEATURE IMPORTANCE & ERROR DIAGNOSTICS & CONFUSION MATRIX
Reviewing loan underwriting accuracy and loss mitigation metrics.


In [9]:
# STEP 11: FEATURE IMPORTANCE & ERROR DIAGNOSTICS
y_pred_rf = best_model.predict(X_test)
print("📋 UNDERWRITING LOAN DEFAULT CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_rf, target_names=['Repaid (Approved)', 'Default Risk']))

cm = confusion_matrix(y_test, y_pred_rf)
print(f"Confusion Matrix:\n{cm}")


📋 UNDERWRITING LOAN DEFAULT CLASSIFICATION REPORT:
                   precision    recall  f1-score   support

Repaid (Approved)       0.92      0.99      0.96      5095
     Default Risk       0.96      0.71      0.81      1422

         accuracy                           0.93      6517
        macro avg       0.94      0.85      0.88      6517
     weighted avg       0.93      0.93      0.92      6517

Confusion Matrix:
[[5048   47]
 [ 417 1005]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & UNDERWRITING SMOKE TEST
Deploying production underwriting pipeline and validating real-time credit decision latency.


In [10]:
# STEP 12 & 13: PRODUCTION SERIALIZATION & LATENCY SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/credit_risk_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_applicant = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_default = loaded_pipe.predict(sample_applicant)[0]
default_prob = loaded_pipe.predict_proba(sample_applicant)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n💳 LIVE LOAN APPLICATION UNDERWRITING AUDIT:")
print(f"   Underwriting Decision: {'🚨 DECLINED (DEFAULT RISK)' if pred_default == 1 else '✅ APPROVED (LOW RISK)'}")
print(f"   Default Probability  : {default_prob*100:.2f}%")
print(f"   Scoring Latency      : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/credit_risk_random_forest_pipeline.joblib (4,447,601 bytes)



💳 LIVE LOAN APPLICATION UNDERWRITING AUDIT:
   Underwriting Decision: ✅ APPROVED (LOW RISK)
   Default Probability  : 9.07%
   Scoring Latency      : 88.962 ms (SLA < 2.0ms: CHECK)
